# EVA Colab Training

**Model**: D=2560, 24 layers, 32 experts, SwiGLU, QK-RMSNorm, RoPE freq=1e6, pos_id binding
**Run**: seq=384, B=1, fp32; profiles — A100 40GB: B=1 + recompute (the M64.9r2 memory-margin choice); L4 22.5GB: B=1 (+ckpt); T4: B=1 +ckpt. Budget horizon 150k steps (up to 67M tokens at B=2 of the 1.93B corpus). Loop policy: only core.adaptation intervenes (M28); cache/depth/highway audits: M31-M35.
**Data**: token_stream_*.bin files in Google Drive; last 3 files = file-level hold-out (audit M13)

---

In [ ]:
# @title 1. Mount Drive & Install Deps
import os, sys, math, time, glob, json, gc

# CUDA allocator: reduce fragmentation on T4 (24 layers x D=2560)
os.environ.setdefault('PYTORCH_CUDA_ALLOC_CONF', 'expandable_segments:True')

from google.colab import drive
drive.mount('/content/drive')

# Config -- point this to your Drive folder with eva_clm/ and data/
DRIVE_ROOT = '/content/drive/MyDrive/eva_clm'  # @param {type:'string'}
DATA_DIR    = os.path.join(DRIVE_ROOT, 'data')
SAVE_DIR    = os.path.join(DRIVE_ROOT, 'checkpoints')
LOG_DIR     = os.path.join(DRIVE_ROOT, 'logs')

os.makedirs(SAVE_DIR, exist_ok=True)
os.makedirs(LOG_DIR, exist_ok=True)

print(f'DRIVE_ROOT={DRIVE_ROOT}')
print(f'DATA_DIR={DATA_DIR}')
print(f'SAVE_DIR={SAVE_DIR}')


In [ ]:
# @title 2. Clone Code from GitHub (data on Drive)
import subprocess

DST = '/content/eva_clm'
if not os.path.exists(DST):
    print('Cloning from GitHub...')
    subprocess.run(['git', 'clone', 'https://github.com/BlackCatSpb/EVA-CLM.git', DST], check=True)
    print('Done.')
else:
    print('Already cloned, force-pulling latest...')
    subprocess.run(['git', '-C', DST, 'fetch', 'origin', 'master'], check=True)
    subprocess.run(['git', '-C', DST, 'reset', '--hard', 'origin/master'], check=True)

GIT_HASH = subprocess.check_output(['git', '-C', DST, 'rev-parse', '--short', 'HEAD']).decode().strip()
print(f'Git HEAD: {GIT_HASH}')

sys.path.insert(0, DST)
os.chdir(DST)
print(f'Working dir: {os.getcwd()}')


In [ ]:
# @title 3. Verify GPU & Imports
import torch
import torch.nn.functional as F
import numpy as np
import math
from torch.serialization import add_safe_globals
from core import EVAConfig, EVAStack, MirrorLRScheduler

add_safe_globals([EVAConfig])

device = 'cuda' if torch.cuda.is_available() else 'cpu'
gpu_name = torch.cuda.get_device_name(0) if device == 'cuda' else 'N/A'
gpu_mem = torch.cuda.get_device_properties(0).total_memory / 1e9 if device == 'cuda' else 0
print(f'Device: {device}  GPU: {gpu_name}  VRAM: {gpu_mem:.1f} GB')

# A100 (Ampere+) only: fp32 matmuls go through TF32 tensor cores (~10-bit
# mantissa, per-op rel.err ~1e-3 — orders below gradient noise and the eval
# spread). On T4/V100 these flags are no-ops for TC-less GPUs (T4 has no
# TF32; the setting is ignored there), so this is safe to leave on.
torch.backends.cuda.matmul.allow_tf32 = True
torch.backends.cudnn.allow_tf32 = True
print(f'TF32: ON   bf16 supported: {torch.cuda.is_bf16_supported() if device == "cuda" else False}')
print(f'PyTorch: {torch.__version__}  CUDA: {torch.version.cuda}')
# M16: enable the expandable-segments allocator for real. Colab's kernel
# imports torch before cell 1 can set the env var, so PYTORCH_CUDA_ALLOC_CONF
# never applied (both live OOM messages kept recommending it). Private-API
# setter; guarded -> older torch / CPU runtimes are a no-op.
try:
    torch.cuda.memory._set_allocator_settings('expandable_segments:True')
    print('cuda allocator: expandable_segments=ON')
except Exception as _ae:
    print(f'cuda allocator: expandable_segments unavailable ({type(_ae).__name__})')


In [ ]:
# @title 4. Build Model (A100 profile: D=2560, 24 layers, seq 384, batch 1, recompute on)
gc.collect()
torch.cuda.empty_cache()

cfg = EVAConfig(
    D=2560,
    n_layers=24,
    bind_K=32,
    code_dim=64,  # B2: twin_free codebook needs K=64 for 65 536 words at overlap≤S−2 (d=D/K=40)
    codebook='twin_free',  # B2: no overlap-5 twins — measured recall knee T 550→≥1200
    vocab=65536,
    mask_eos=False,  # do not mask EOS (separate files *_eos.bin)
    mlp_groups=32,
    mlp_expand=4,
    batch_size=1,  # M64.9r2 (operator): B=1 — the M64 telemetry (census +
                  # measure_kill's retained traversals) pushed the reserved to
                  # 36.6/40 at d=12; B=1 halves the activation memory and buys
                  # the margin for the depth ladder (16/20/24).
    seq_len=384,  # M64.9r2 (operator): 12*CHUNK(32)=384; the mid-layer tau3=128
                  # now gets 3 tau of resolution (was 1.75 at 224); the inter-window
                  # carry on tau4=512 is exp(-384/512)=0.47 (was 0.65); the cache
                  # attention 12x384=4608 keys (fits with B=1). NOTE: the val is
                  # NOT directly comparable with the 224-window runs — compare slopes.
    lr=6e-4,  # M60: the unigram arithmetic (3e-4 needed ~10-17k steps for
             # the token frequencies; 6e-4 halves it, the guards carry the risk)
    max_steps=150_000,  # ~38M tokens = 2% of the 1.93B corpus in ONE Colab-Pro budget cycle (~160-190h of ~307h left);
                                # the run is resumable — cursor/RNG/val-history all ride in best.pt (M12)
    warmup_steps=1200,  # NOTE: __post_init__ пересчитывает warmup/log/eval из lambda-дома — см. пост-init ниже
    log_interval=55,
    eval_interval=1045,  # NOTE: тоже перетирается __post_init__ — см. ниже
    optimizer='eva_proj',  # A2 arm: EVAAdamW + AdamP-projection. A0='adamw', A1='eva'
    save_interval=1000,
    scheduler='mirror',
    lr_boost_max=2.0,  # upward LR path: allow LR to climb above base when val on downtrend (0=disable)
    lr_improve_tol=0.002,  # downtrend tolerance for the boost gate (hysteresis vs eval noise)
    per_layer_ls_lr=True,  # per-layer LR modulation from fast/slow EMA var(log_scale)
    ls_ema_fast=0.99,
    ls_ema_slow=0.999,
    ls_mult_min=0.5,
    ls_mult_max=2.0,
    ls_mirror_mult_max=2.0,
    private_mem=True,
    expert_asymmetry=True,
    meta_trust=True,
    data_dir=DATA_DIR,
    save_dir=SAVE_DIR,
    log_dir=LOG_DIR,
    conv_kernel=48,
    gradient_checkpointing=True,  # A100: ON at B=2 (measured no-ckpt math above). Costs ~30%, buys the margin that keeps EVAL/ggeo from OOMing the ladder into a sawtooth. A no-ckpt run is fine at B=1 (~60-70 tok/s on A100; B=2+recompute ~90). Caveat M35-era: recompute re-runs the forward, so the manual i-gate noise is drawn twice — gradients stay finite, just not bit-exact to the stored activations.
    head_mode='sigmoid_coded',
    embed_center=False,  # B9 (audit 02a): kills codebook common-mode (pair-cos 0.951->0.000, roundtrip 1.000). GEOMETRY CHANGE — set True on a FRESH run only.
    head_normalize=True,
    head_u_wall=1e-3,       # M52a: soft wall on the bit log-odds (the
                          # saturation escape; 0 disables)
    head_phantom_bits=32,   # M52b: phantom basis rank (the lacuna channel)
    head_phantom_noise=0.05,  # M52b: exploration-noise init (learnable)
    head_srl=True,          # M53: State Resolution Loop in the forward
    head_srl_steps=2,       # M53: gentle refinement (2 passes)
    head_srl_after=1045,    # M53c: SRL joins after the first eval (at init
                            # it commits to random codes and doubles the CE)
    head_phantom_slots=16,  # M54: the phantom-concept bank
    ucl_read_scale_floor=0.1,      # M59: let the UCL prove itself before the
    ucl_read_scale_floor_until=5000,  # model self-closes it (measured -4.0 in
                                     # 120 steps); released after 5k steps
    head_phantom_max=64,    # M59c: the phantom channel capacity (grows in place)
    stream_chunk_steps=1000,  # M64.9 (M63-F): 250 steps = 112k tokens was ~30 s of
                              # training per genre — the CE cannot learn even the genre
                              # unigram in that window. 1000 steps = 448k tokens is the
                              # stability/novelty compromise; the novelty machinery
                              # still gets ~7 shifts per 7.3k steps. 250 remains a
                              # limited A/B arm; 0 = the legacy exhaustion-only cadence.
    bind_twist_mode='trajectory_spiral',
    bind_traj_dims=3,
    hybrid_alpha_max=0.7,
    hybrid_alpha_min=0.3,
    w_pred_scale_init=3.0,
    bind_twist_gate=True,
    collective_layer=True,
    collective_layer_idx=None,
    collective_read_out=True,
    collective_uncert_theta=0.5,
    collective_uncert_kappa=3.0,
    collective_contra_thresh=-0.1,
    collective_contra_gain=6.0,
    collective_maturity_thresh=0.12,
    surprisal_weight=0.3,
    branch_balance_weight=0.1,
    variable_precision=True,
    precision_threshold=0.3,
    explicit_reasoning=True,
    reasoning_max_steps=8,
    reasoning_adaptive=True,  # adaptive-depth reasoning loop (gated by knowledge signals)
    use_amp=False,  # cycle-1 fp32 clean baseline; bf16 crash fixed in B10b (UCL dtype) — flip True after the first EVAL for ~2x on A100
    intent_bridge=True,  # Intent Bridge: top-down/bottom-up intent signal (unbounded context)
    bridge_glu=True,  # BridgeGLU: live semantic MLP gate (replaces frozen mod_scale_mlp; no boost hack needed)
    bridge_conn=0.1,  # aux: bridge learns connectivity by predicting next token
    maturation_enabled=True,  # unified wake-up gate: live mod / mem-write / bridge-inject / intent gated by layer maturity
    pm_write_delay=0,  # maturity-only write gate (no random-regime echo floor)
    intent_topdown=True,  # top-down intent_state propagation
    memory_bank=True,  # Streaming Memory Bank: L1 buffer + L2 bank + L3 concepts
    mem_l1_slots=3,  # L1: rolling buffer of last 3 sentences (immediate)
    mem_l2_slots=32,  # L2: learned bank with 32 slots (short-term)
    mem_min_write_mat=0.3,  # min maturation before writes allowed (like private_mem)
    mem_bridge_dim=256,  # memory bank bridge dim (matches bridge_dim)
    concept_birth_novelty_threshold=0.15,  # birth only if d_min > threshold
    unified_concept_layer=True,  # unified concept layer (global, after embedding)
    unified_concept_S=8,  # number of concept prototypes
    logit_cache_enabled=True,  # decision #3: code-space cache wired into forward (bias -10 gate => true identity at init)
    logit_cache_max_entries=12,  # A100/L512: window 12x512=6144 keys == same attention load as the old 32x256 (T4 sizing OOMed at 32x384=12288: 33.6GB)
    logit_cache_n_heads=8,  # attention heads for logit cache
    logit_cache_scheduled_sampling=0.05,  # R1: 5% inference-mode during training
    logit_cache_reset_on_resume=True,  # R6: clear cache on resume/LR-reset
    orth_weight=0.0,  # ortho-gram aux off (32x 2560^2 gram graphs are huge)
    div_weight=10.0,  # sigmoid-bounded log_scale divergence
)

# EVAConfig.__post_init__ пересчитывает warmup_steps/log_interval/eval_interval
# из lambda-дома (lambda_utils.LambdaConfig) и ЗАТИРАЕТ аргументы конструктора.
# Поэтому фиксируем явно ПОСЛЕ создания cfg — одинаково для всех рук A/B
# (иначе lambda-дом даёт warmup~101, eval~233, что для 723M слишком резко).
cfg.warmup_steps = 300   # M60: 1200 -> 300
cfg.aux_kill_switch = True   # M64.12: MEASURE-ONLY telemetry (the proj values
                              # in the ks: line); the disable mode stays off until
                              # the measure-only calibration (see the whiteboard)
cfg.balancer_align_every = 8  # M64.4: the align path costs 3 graph traversals
                              # (CE/aux/bypass; ~3 recomputes with checkpointing) —
                              # the largest structural cost of the run. Align every
                              # 8th step, the cheap one-backward normalized total
                              # otherwise -> 1.25 traversals/step (0 = never align).
cfg.log_interval = 55
cfg.eval_interval = 440  # оператор: каждые 440 = 8*55 (выравнивание оценок с логом)
# M64.7-разморозка readout (оператор 2026-09-19): снять λ⁻²-демпфер с
# embed.basis/lm_head.readout — голова = бутылочное горло (bias-decomp 95.8%).
cfg.readout_lr_mult = 1.0
# T9.8: low-rank K/V-пространство кэша логитов (0 = полное, бит-совместимо).
# Экономия: K/V-кольцо 94.4→2.4MB, attention-параметры 29.0M→0.73M.
cfg.logit_cache_kv_dim = 64
# T9.15 (A/B-рука, оператор): многоразрешающий кэш — пулы K/V на фиксированных
# шкалах tau (сборка снизу вверх от атомов 8; модель выбирает разрешение через
# обучаемые level-эмбеддинги). Пусто = выключено (текущее поведение). Пример
# включения: cfg.logit_cache_ms_spans = "8,32,128,512"; ms_max — кольцо на шкалу.
cfg.logit_cache_ms_spans = "8,32,128,512,2048,8192"
cfg.logit_cache_ms_max = 16
# T9.7b: лестница салиентности головы = лестница кэша (выравнивание горизонтов
# VSA/кэш/салиентность), гейты читают свои τ-шкалы (128 — рабочая, base — самая
# длинная). Порог наблюдения фантома — самокалибрующийся (median + k*MAD), а не
# легаси-константа 1.1, которая стояла выше всего диапазона сигнала (p99~1.08).
cfg.head_lacuna_ladder = tuple(float(s) for s in str(cfg.logit_cache_ms_spans).split(',') if s)
# T9.7b: порог наблюдения фантома — самокалибрующийся: 1 + max(floor, k*MAD)
# над базовой ставкой (MAD кольца салиентности). Не квантиль (нет гарантии
# срабатываний) и не легаси-константа 1.1 (она стояла выше всего диапазона).
cfg.head_phantom_thr_mode = 'noise'
cfg.head_phantom_thr_k = 2.0
cfg.head_phantom_thr_floor = 0.002
# P1-2 (аудит): парный канал головы (rank-r Ising) — A/B-рука, identity на старте
# (pair_V2=0 ⇒ побитово прежний forward). Включать на резюме можно: параметры новые,
# оптимизатор их просто пропустит при восстановлении по именам.
# cfg.head_pair_rank = 16

# ─── P4 (по заявке автора): прямая работа с состояниями. Все руки default off,
#     чекпойнт-совместимы (zero-init/флаги/non-persistent буферы); новые
#     параметры оптимизатор пропускает по именам (fresh Adam). ───
# P4-3: детектор межшкального противоречия (fast vs slow VSA) — МОНИТОР:
# forward не меняет, даёт chi_time/chi_ladder в телеметрию. Включать безопасно.
cfg.contradiction_field = True
# P4-3 A/B-руки потребителей (меняют логиты — включать по одной, следить за val):
# cfg.head_temper_rel = True       # tempering по относительному χ (лестница)
# cfg.phantom_chi_salience = True  # противоречие как новизна для наблюдения фантома
# P4-1 A/B: обучаемая добавка к гейту зеркала (общий InnerEye, ~0.5k параметров,
# zero-init ⇒ identity на старте). Метрика A/B: gate_selectivity + CE за 500+ шагов.
# cfg.inner_eye = True
# P4-2: зонд читаемости внутренних сигналов из h (MetaHead, ~329k параметров).
# Тренирует ТОЛЬКО свою голову (meta_head_grad=False, h.detach) — ствол не тронут.
# cfg.meta_head = True
# P4-2 подписной эксперимент (офлайн на чекпойнте, GPU прогона не трогает):
#   python scripts/probe_introspection.py --ckpt checkponts/best.pt \
#       --file data/holdout.bin --seq 384 --steps 256 --inject-token 6 --inject-latent 4
print(f'Overrides: warmup={cfg.warmup_steps}, log_interval={cfg.log_interval}, eval_interval={cfg.eval_interval}, readout_lr_mult={cfg.readout_lr_mult}, logit_cache_kv_dim={cfg.logit_cache_kv_dim}')

# Reproducibility (M56b): the model init was UNSEEDED - every restart began
# from a different random point (the step-0 ce_raw varied 11.8-15.3 across
# otherwise identical runs), which makes A/B arms incomparable. The data
# rng is already seeded (42) in the streams cell.
torch.manual_seed(1234)
model = EVAStack(cfg).to(device)
n_params = model.param_count()
print(f'Model: {n_params:,} params ({n_params/1e6:.2f}M)')
print(f'Head: {cfg.head_mode} (normalize={cfg.head_normalize}) | M52-54: '
      f'wall={cfg.head_u_wall} Kp={cfg.head_phantom_bits} '
      f'srl={cfg.head_srl}({cfg.head_srl_steps},from {cfg.head_srl_after}) '
      f'ph_slots={cfg.head_phantom_slots} ph_from={cfg.head_phantom_after} '
      f'ph_thr={cfg.head_phantom_thr} ph_max={cfg.head_phantom_max}')
print(f'  M59: ucl_floor={cfg.ucl_read_scale_floor} until={cfg.ucl_read_scale_floor_until} '
      f'(links A/B/C active: phantom<->UCL)')
print(f'  M62: stream_chunk={cfg.stream_chunk_steps} steps '
      f'(~{cfg.stream_chunk_steps * cfg.batch_size * cfg.seq_len / 1000:.0f}k tokens/chunk), '
      f'ph_decay={cfg.head_phantom_decay}')
print(f'  M55a: temper={cfg.head_temper}(k={cfg.head_temper_k},cos={cfg.head_temper_cos},'
      f'from {cfg.head_temper_after}) mem_lacuna_k={cfg.mem_lacuna_k}')
print(f'Bind: {cfg.bind_twist_mode} (gate={cfg.bind_twist_gate})')
print(f'Collective: maturity={cfg.collective_maturity_thresh}, read_out={cfg.collective_read_out}')
print(f'Variable Precision: {cfg.variable_precision} (threshold={cfg.precision_threshold})')
print(f'Explicit Reasoning: {cfg.explicit_reasoning} (max_steps={cfg.reasoning_max_steps}, adaptive={cfg.reasoning_adaptive})')
print(f'AMP: {cfg.use_amp}')

gc.collect()
torch.cuda.empty_cache()


In [ ]:
# @title 5. Precision policy (AMP) — batch & window live in cell 4
# M36: this cell carried a T4-era leftover `cfg.batch_size = 1` that SILENTLY
# trampled whatever cell 4 declared (every B>1 profile since was quietly
# running B=1 — found in the full-notebook audit, 2026-09-13). Cell 4 is now
# the single source for batch/seq/ckpt.
print(f'Batch size: {cfg.batch_size}')
print(f'  Tokens/step: {cfg.batch_size * cfg.seq_len}')

# AMP policy (A100): cfg.use_amp=True in cell 4 enables bf16 autocast around
# the FORWARD (tensor cores + big activation-RAM saving — the real reason the
# seq-512 run sits at 74% VRAM). fp16 deliberately NOT used: the collective
# layer overflows it (that's why T4 ran fp32). bf16 needs no GradScaler:
# exponent range == fp32. Loss ops (CE/BCE) are on autocast's fp32 list and
# are promoted automatically, so the coded-head NLL keeps full precision.
_USE_AMP = bool(getattr(cfg, 'use_amp', False)) and device == 'cuda'
_AMP_DTYPE = torch.bfloat16 if (_USE_AMP and torch.cuda.is_bf16_supported()) else torch.float16
_USE_AMP = _USE_AMP and _AMP_DTYPE is torch.bfloat16  # fp16 fallback disabled by design
scaler = None  # bf16: no loss scaling needed


In [ ]:
# @title 6. Optimizer mode (A/B arms)
# NOTE: the real optimizer is built in cell 8b via core.adaptation.build_optimizer
# with optimizer=getattr(cfg, 'optimizer', 'adamw'). This cell only validates the mode.

import torch
from core.adaptation import build_optimizer

_mode = getattr(cfg, 'optimizer', 'adamw')
assert _mode in ('adamw', 'eva', 'eva_proj'), f'unknown optimizer={_mode!r}'
_probe = build_optimizer(model, cfg.lr, llrd_decay=1.0,
                         weight_decay=cfg.weight_decay, betas=(0.9, 0.95),
                         optimizer=_mode)
if _mode in ('eva', 'eva_proj'):
    print(f'Optimizer: EVAAdamW mode={_probe.defaults["mode"]} '
          f'projected={_probe.defaults["projected"]} '
          f'groups={len(_probe.param_groups)}')
else:
    print(f'Optimizer: torch.optim.AdamW groups={len(_probe.param_groups)}')
del _probe, _mode
print(f'cfg.optimizer = {cfg.optimizer} (A0=adamw | A1=eva | A2=eva_proj)')


In [ ]:
# @title 7. Data Streams
class TokenStream:
    def __init__(self, path):
        self.data = np.memmap(path, dtype=np.uint16, mode='r')
        self.path = path
        self.len = len(self.data)
    def get_batch(self, seq_len, batch_size, offset, vocab=None):
        needed = batch_size * seq_len + 1
        wrapped = offset + needed > self.len
        if wrapped:
            offset = 0
        chunk = self.data[offset:offset + needed]
        # B7 (audit 01 F-01): loud corpus/vocab mismatch, no silent clip.
        if vocab is not None:
            _hi = int(chunk.max(initial=0))
            if _hi >= vocab:
                raise ValueError(f'TokenStream: id {_hi} >= vocab {vocab} in {self.path} - corpus/model mismatch')
        x = torch.from_numpy(chunk[:batch_size * seq_len].reshape(batch_size, seq_len).copy())
        y = torch.from_numpy(chunk[1:batch_size * seq_len + 1].reshape(batch_size, seq_len).copy())
        # Audit M8: explicit `wrapped` flag — the old 3-tuple let get_batch
        # silently rewind the offset, so the caller's rotation branch (new
        # stream + document-state reset) only ever ran at step 0 and one
        # stream served the whole run.
        return x.long(), y.long(), offset + batch_size * seq_len, wrapped

stream_files = sorted(glob.glob(os.path.join(DATA_DIR, 'token_stream_*_clean.bin')))
if not stream_files:
    stream_files = sorted(glob.glob(os.path.join(DATA_DIR, 'token_stream_*.bin')))
if not stream_files:
    print('WARNING: No token_stream_*.bin found!')
    print(f'  Looked in: {DATA_DIR}')
    print('  Using random data for testing')
    streams = []
else:
    streams = [TokenStream(f) for f in stream_files]
    total_tokens = sum(s.len for s in streams)
    print(f'Found {len(streams)} files, {total_tokens:,} total tokens')
    # Audit M13: the M8 hold-out was ONE file — val_loss was a single
    # domain-draw of the corpus. With 39 files, hold out the last three
    # (file-level, never trained on) and average val over them; per-file
    # batch budget is divided so total eval cost stays ~constant. Small
    # corpora (< 8 files) keep the M8 single-stream semantics.
    _hold_n = 3 if len(streams) >= 8 else (1 if streams else 0)
    if streams:
        _hold_files = [os.path.basename(f) for f in stream_files[-_hold_n:]]
        print(f'  train pool: {len(streams) - _hold_n} files | hold-out: {_hold_n} files: '
              + ', '.join(n[-28:] for n in _hold_files))


In [ ]:
# @title 8. Checkpoint (fresh start for A/B)
# B15 (audit 05 F5-02): verify_identity_resume/codebook_fingerprint are used
# HERE but first imported in cell 9 — fresh runtime + FORCE_FRESH=False died
# with NameError. Import at point of need (idempotent).
from core.training_control import codebook_fingerprint, verify_identity_resume
def _t15(o):
    # B15 (F5-07): move restored streaming state (nested tensors) to device
    if torch.is_tensor(o):
        return o.to(device)
    if isinstance(o, (list, tuple)):
        return [_t15(x) for x in o]
    return o
FORCE_FRESH = False  # operator: РЕЗЮМ с best.pt (T9.10-сейв проверен; Drive-best
                     # восстановлен на 7040/8.5824). True = свежий прогон С НУЛЯ —
                     # ставить только для нового A/B (многоразрешающий кэш должен
                     # работать с первых окон, а не прикручиваться к обученной голове).
                     # База сравнения свежего прогона — записанная траектория 4725d59.
start_step = 0
# B16: FORCE_FRESH must not inherit stale resume vars from a prior run
_resume_cuda_rng = None
resumed_active_depth = None  # set on resume; None for fresh start / pre-fix ckpt
state = None
best_val_loss = float('inf')
# M12 single-best.pt: everything a restart needs rides in the checkpoint;
# these hold the restored values (defaults = fresh-start cursor).
resumed_stream_idx = 0
resumed_offset = 0
_resume_balancer_sd = None
_resume_rng = None
_resume_data_rng = None

best_ckpt = os.path.join(SAVE_DIR, 'best.pt')

# Find the BEST numbered checkpoint (best N.pt) by step number.
# best.pt may be overwritten with step=0 data; numbered files are immutable.
def _find_best_checkpoint(save_dir):
    import re
    numbered = []
    for f in glob.glob(os.path.join(save_dir, 'best [0-9]*.pt')):
        m = re.search(r'best\s+(\d+)\.pt$', os.path.basename(f))
        if m:
            try:
                ckpt = torch.load(f, map_location='cpu', mmap=True, weights_only=False)
                step = ckpt.get('step', 0)
                val = ckpt.get('best_val_loss', float('inf'))
                numbered.append((f, step, val))
            except Exception:
                pass
    if numbered:
        # Prefer highest step, then lowest val_loss
        numbered.sort(key=lambda x: (-x[1], x[2]))
        return numbered[0]
    return None

# Single-checkpoint policy: prefer best.pt if it has real progress;
# fall back to numbered checkpoints if best.pt is stale (step=0).
ckpt_files = []
# Checkpoint name policy (operator): best.pt ONLY, as before M42 — the
# rolling M42 file is gone (the poisoned step-2970 state was auto-resumed
# once and cost a whole session). best.pt is the freshest val-clean state.
if os.path.exists(best_ckpt) and not FORCE_FRESH:
    try:
        _ckpt = torch.load(best_ckpt, map_location='cpu', weights_only=False)
    except RuntimeError as e:
        print(f'  best.pt CORRUPTED ({e}), falling back to numbered checkpoints...')
        _ckpt = {'step': 0}
    if _ckpt.get('step', 0) > 0:
        ckpt_files = [best_ckpt]
        # T9.10b: an immutable numbered copy can be NEWER than best.pt when a
        # FUSE-swallowed save left best.pt stale (observed: best.pt=6600 while
        # val_history best=8.5824@7040). Prefer the higher step.
        _best_step_now = _ckpt.get('step', 0)
        _num_res = _find_best_checkpoint(SAVE_DIR)
        if _num_res and _num_res[1] > _best_step_now:
            print(f'  T9.10b: numbered {os.path.basename(_num_res[0])} (step={_num_res[1]}) is newer than best.pt (step={_best_step_now}) - preferring it')
            import shutil
            shutil.copy2(_num_res[0], best_ckpt)
            ckpt_files = [best_ckpt]
    else:
        if not ckpt_files:
            print(f'  best.pt has step=0 (stale or corrupted), looking for numbered checkpoints...')
        result = _find_best_checkpoint(SAVE_DIR)
        if result:
            best_ckpt, _step, _val = result
            ckpt_files = [best_ckpt]
            print(f'  Found best numbered checkpoint: {os.path.basename(best_ckpt)} (step={_step}, val={_val:.4f})')
            import shutil
            shutil.copy2(best_ckpt, os.path.join(SAVE_DIR, 'best.pt'))
            print(f'  Copied to best.pt')
    del _ckpt

if (not FORCE_FRESH) and ckpt_files:
    # Prefer FULL (uncompressed) checkpoints so optimizer/scheduler survive resume.
    # FCF-CPR compressed checkpoints STRIP optimizer state -> fresh Adam -> instability.
    def _is_full(p):
        s = os.path.basename(p)[:-3]  # drop '.pt'
        return ('_fcf' not in s) and ('_cpr' not in s)
    full = [p for p in ckpt_files if _is_full(p)]
    if full:
        latest = full[-1]
        if latest != ckpt_files[-1]:
            print(f'  Preferring full checkpoint {latest} over newer compressed one (keeps optimizer state)')
    else:
        latest = ckpt_files[-1]
        if any(('_fcf' in os.path.basename(p)) or ('_cpr' in os.path.basename(p)) for p in ckpt_files):
            print('  WARNING: resumed from COMPRESSED checkpoint - optimizer/scheduler state absent -> fresh Adam, unstable. Use a full step_*.pt for clean resume.')
    print(f'Resuming from {latest}')
    ckpt = torch.load(latest, map_location=device, weights_only=False)
    resumed_active_depth = ckpt.get('active_depth', None)  # restore true depth if present
    from core.migrate import migrate_state_dict
    sd, n_mig = migrate_state_dict(dict(ckpt['model']), model)
    if n_mig:
        print(f'  MIGRATED {n_mig} keys (W_out +K, bind_coh_gate=0, freq_scale=1.0)')
    # T9.15b: лестница шкал выросла (2048 -> +8192): дописываем свежие
    # level-строки В КОНЕЦ (row 0 = база, rows 1..k = шкалы — сохраняются),
    # чтобы выученные адреса шкал не потерялись (иначе size-mismatch -> SKIP).
    _msd8 = model.state_dict()
    for _k8 in [k for k in sd if k.endswith('ms_emb.weight') and k in _msd8]:
        if sd[_k8].shape[0] < _msd8[_k8].shape[0]:
            _extra8 = torch.zeros(_msd8[_k8].shape[0] - sd[_k8].shape[0],
                                  sd[_k8].shape[1], dtype=sd[_k8].dtype,
                                  device=sd[_k8].device)
            torch.nn.init.normal_(_extra8, std=0.02)
            print(f'  T9.15b: {_k8} {list(sd[_k8].shape)} -> '
                  f'{[_msd8[_k8].shape[0], sd[_k8].shape[1]]} (appended new scale rows)')
            sd[_k8] = torch.cat([sd[_k8], _extra8], dim=0)
    # Filter size-mismatched keys (e.g. L2 slots changed 16->32)
    _model_sd = model.state_dict()
    _skipped = []  # B8: identity drift is fatal
    _filtered = {}
    for k, v in sd.items():
        if k in _model_sd and _model_sd[k].shape != v.shape:
            print(f'  SKIP size-mismatch: {k} ckpt={list(v.shape)} model={list(_model_sd[k].shape)}')
            _skipped.append(k)
        else:
            _filtered[k] = v
    miss, unex = model.load_state_dict(_filtered, strict=False)
    verify_identity_resume(model, ckpt, _skipped)  # B8 (02a F2A-03)
    _resume_cuda_rng = ckpt.get('cuda_rng')   # B16 (applied in cell 8 directly)
    # B15 (audit 05 F5-07): mid-document resume continues the DOCUMENT —
    # per-layer VSA state + global_state used to live outside best.pt, so every
    # Colab restart cold-started the current document (measured 0.37 nat).
    # B16 (audit 05 follow-up): the captured state is a LIST holding
    # tensors — truthiness must go through len(), bool(list_of_tensors)
    # evaluates the tensor ('Boolean value of Tensor ... ambiguous').
    if ckpt.get('stream_state') is not None and len(ckpt['stream_state']):
        state = _t15(ckpt['stream_state'])
    if ckpt.get('stream_gs') is not None and len(ckpt['stream_gs']):
        gs = _t15(ckpt['stream_gs'])
    if isinstance(ckpt.get('intent_stream'), list):   # T7: тёплый resume intent-потока
        model._intent_stream = _t15(ckpt['intent_stream'])

    # fix: reopen cognitive gate on resume — hybrid_gate (sigmoid+softmax)
    # replaces frozen mod_scale_mlp. Initialize tau for per-expert specialization.
    # B15 (F5-09): reopen only for checkpoints that PREDATE hybrid_gate —
    # it fired every resume and clobbered the RESTORED learned gate bias and
    # log_tau with config defaults.
    if getattr(cfg, 'mlp_gate_b_init', 0.0) > 0 and not any(k.endswith('mlp_gate_b') for k in _filtered):
        tau_val = getattr(cfg, 'mlp_hybrid_gate_tau', 1.0)
        for layer in model.layers:
            layer.mlp.mlp_gate_b.data.fill_(cfg.mlp_gate_b_init)
            layer.mirror.hybrid_gate.log_tau.data.fill_(math.log(tau_val))
        print(f'  reopened cognitive gate: mlp_gate_b -> {cfg.mlp_gate_b_init}, hybrid_gate tau -> {tau_val:.3f}')
        if getattr(cfg, 'bridge_glu', False):
            print('  bridge_glu=True: BridgeGLU params (bridge_glu_net.*) absent in old ckpts -> start fresh, will adapt.')

    # fix: private-mem write step not saved in pre-fix checkpoints.
    # _pm_step is a conditional buffer (exists only when cfg.private_mem is on),
    # so guard the attribute — private_mem=False models never had it and must
    # not crash on resume (this path only fires when FORCE_FRESH=False).
    pm_key = 'layers.0.mirror._pm_step'
    if pm_key not in ckpt['model'] and hasattr(model.layers[0].mirror, '_pm_step'):
        for l in model.layers:
            if hasattr(l.mirror, '_pm_step'):
                l.mirror._pm_step.fill_(5000)
        print('  _pm_step not saved (pre-fix checkpoint): private-mem write enabled now')

    if miss:
        print(f'  Missing keys: {len(miss)}')
    if unex:
        print(f'  Unexpected keys: {len(unex)}')

    def _restore_optimizer(optimizer, model, ckpt):
        ckpt_opt = ckpt.get('optimizer', {}) or {}
        old_names = ckpt.get('param_names')
        if old_names is None:
            old_names = ckpt_opt.get('param_names')  # legacy location
        if old_names is None:
            print('  WARNING: no param_names - optimizer state NOT restored (fresh Adam)')
            return False
        names = {id(p): n for n, p in model.named_parameters()}
        pos = {id(p): i for i, p in enumerate(
            (p for g in optimizer.param_groups for p in g['params']))}
        new_sd = optimizer.state_dict()
        new_sd['state'] = {}
        old_state = ckpt_opt.get('state', {})
        old_groups = ckpt_opt.get('param_groups', [])
        moved = skipped = 0
        for name, p in model.named_parameters():
            if name not in old_names:
                skipped += 1
                continue
            si = old_names.index(name)
            st = old_state.get(str(si)) if str(si) in old_state else old_state.get(si)
            if st is None:
                continue
            if tuple(st['exp_avg'].shape) != tuple(p.shape):
                if (name.endswith('bind.W_out') and len(st['exp_avg'].shape) == 2
                        and st['exp_avg'].shape[1] == p.shape[1]
                        and st['exp_avg'].shape[0] < p.shape[0]):
                    st = {k: (v[:p.shape[0]] if isinstance(v, torch.Tensor) and v.dim() == 2 else v)
                          for k, v in st.items()}
                    moved += 1
                else:
                    skipped += 1
                    continue
            else:
                moved += 1
            new_sd['state'][pos[id(p)]] = {k: (v.clone() if isinstance(v, torch.Tensor) else v)
                                            for k, v in st.items()}
        for gi in range(min(len(new_sd['param_groups']), len(old_groups))):
            if 'lr' in old_groups[gi]:
                new_sd['param_groups'][gi]['lr'] = old_groups[gi]['lr']
        optimizer.load_state_dict(new_sd)
        print(f'  Optimizer restored by name: {moved} slots, {skipped} skipped')
        return moved > 0
    start_step = ckpt.get('step', 0)
    best_val_loss = ckpt.get('best_val_loss', float('inf'))
    reasoning_enabled_step = ckpt.get('reasoning_enabled_step', 0)
    _resume_depth_state = ckpt.get('depth_state')  # B14 (F4-06)
    _resume_balancer_sd = ckpt.get('balancer')
    _resume_branch_var_ref = ckpt.get('branch_var_ref')  # M58c (M51 anchor)
    _resume_rng = ckpt.get('rng')
    _resume_data_rng = ckpt.get('data_rng')
    resumed_stream_idx = int(ckpt.get('stream_idx', 0) or 0)
    resumed_offset = int(ckpt.get('offset', 0) or 0)
    print(f'  Resumed at step {start_step}, reasoning ramp t={reasoning_enabled_step}')
else:
    reasoning_enabled_step = 0
    print('No checkpoint found, starting fresh')


In [ ]:
# @title 8b. Adaptation module (unified, principled -- core.adaptation)
# Single source of truth for training stability. Replaces the old scattered
# guard: progressive depth (validation-plateau driven, not a fixed schedule),
# LR (linear warmup + mirror-adaptive multiplier + plateau damping + rewind on
# recovery), statistical failure detection (3 sigma CE rule), adaptive gradient
# clipping (AGC), and aux-loss balancing via spectral gradient alignment
# (no per-loss magic weights).
from core.adaptation import (LossBalancer, DepthController, LRController,
                             GradientClipper,
                             codebook_fingerprint, verify_identity_resume,
                             apply_tau_lr,
                             set_active_depth, build_optimizer)
from core.training_control import grad_census, training_telemetry  # M64.8

# Guard: cell '8. Resume Checkpoint' MUST run before this one. It loads best.pt
# and defines start_step / resumed_active_depth / best_val_loss / ckpt_files /
# reasoning_enabled_step. Running this cell alone (e.g. after a kernel restart)
# leaves them undefined -> the cryptic "NameError: start_step" and, worse, a
# silent fresh start from step 0 (which we never want; we always resume best.pt).
for _need in ('start_step', 'resumed_active_depth', 'best_val_loss', 'ckpt_files',
              'reasoning_enabled_step', 'FORCE_FRESH', 'best_ckpt',
              'resumed_stream_idx', 'resumed_offset',
              '_resume_balancer_sd', '_resume_rng', '_resume_data_rng'):
    if _need not in globals():
        raise RuntimeError(
            "Cell '8. Resume Checkpoint' was not executed. Run it BEFORE this cell "
            "(or Runtime -> Restart session -> Run all). It loads best.pt and defines the needed state.")


def _make_opt(lr):
    cfg.lr = float(lr)
    # llrd_decay=1.0: disable index-based LLRD (replaced by tau_config.lr_mult in training loop)
    # optimizer mode flows from cfg.optimizer ('adamw' | 'eva' | 'eva_proj')
    return build_optimizer(model, lr, llrd_decay=1.0,
                           weight_decay=cfg.weight_decay, betas=(0.9, 0.95),
                           optimizer=getattr(cfg, 'optimizer', 'adamw'),
                           readout_lr_mult=float(getattr(cfg, 'readout_lr_mult', 0.0) or 0.0))  # M64.7

# 3) Progressive unfreeze driven by validation-loss plateau (diminishing
#    returns), NOT a fixed schedule. On resume continue from the checkpoint
#    depth so all-24 never wake at once on a memory-tight T4.
starting_fresh = FORCE_FRESH or not ckpt_files

# 1) Optimizer — restore from checkpoint if available, else fresh Adam.
optimizer = _make_opt(cfg.lr)
if not starting_fresh and '_restore_optimizer' in dir():
    # Use name-based restore (handles architecture changes between saves)
    if not _restore_optimizer(optimizer, model, ckpt):
        print('  WARNING: optimizer state not restored (fresh Adam)')
elif not starting_fresh and 'optimizer' in ckpt and ckpt.get('optimizer') is not None:
    try:
        optimizer.load_state_dict(ckpt['optimizer'])
        print('Optimizer restored from checkpoint (momentum preserved)')
    except Exception as e:
        print(f'[warn] optimizer restore failed: {e} — using fresh optimizer')
else:
    print('Optimizer: fresh (no checkpoint state)')

# 2) LR controller: warmup + mirror-adaptive multiplier + plateau damping.
#    Restore full scheduler state (EMA baselines, val_ema, loss_lr_factor)
#    to avoid LR instability after resume.
scheduler = LRController(model, optimizer, cfg=cfg)
if not starting_fresh and 'scheduler' in ckpt and ckpt.get('scheduler') is not None:
    scheduler.load_state_dict(ckpt['scheduler'])
    print(f'Scheduler state restored (step={start_step})')
else:
    scheduler.set_step(start_step)
    print(f'Scheduler: fresh (step={start_step})')

if starting_fresh:
    init_k = 8
else:
    # Restore the TRUE achieved active depth from the checkpoint (not the
    # step-based heuristic) so frequent stop/resume doesn't keep resetting to
    # 8 active layers (which would freeze deep layers forever). Falls back to
    # the legacy heuristic for pre-fix checkpoints that lack 'active_depth'.
    init_k = resumed_active_depth if resumed_active_depth is not None \
        else min(8 + (start_step // 15000) * 4, cfg.n_layers)
depth = DepthController(model, n_layers=cfg.n_layers, init_k=init_k,
                        unfreeze_inc=4, eval_interval=cfg.eval_interval)
set_active_depth(model, init_k)
print(f'Progressive unfreeze: init_active={init_k} (fresh={starting_fresh}, start_step={start_step})')

# 4) Aux-loss balancer: spectral alignment bounds the aux gradient by ||g_CE||.
balancer = LossBalancer(align=True, align_cap=10.0, eval_interval=cfg.eval_interval,
                        align_every=int(1 if getattr(cfg, 'balancer_align_every', 1) is None
                                        else getattr(cfg, 'balancer_align_every', 1)),  # M64.4
                        kill_terms=list(LossBalancer.AUX_TERMS)
                        if getattr(cfg, 'aux_kill_switch', False) else None,  # M64.12
                        kill_disable=bool(getattr(cfg, 'aux_kill_disable', False)),
                        safety_aux=('head_wall',) if getattr(cfg, 'balancer_safety_ungated', True)
                        else ())  # T9.6: стена головы вне CE-гейтинга (A/B-ручка)

# B14 (F4-06): resume the plateau integrator (depth exists by now, cell-9 order).
depth.put_state(_resume_depth_state if '_resume_depth_state' in dir() else None)
# M58c: a resumed session is a regime change — the balancer baselines and
# the M51 branch-anchor reference must come back from the checkpoint.
balancer.load_state_dict(_resume_balancer_sd)
_resume_branch_var_ref = globals().get('_resume_branch_var_ref')  # M58d: stale cell 8 OK
if _resume_branch_var_ref is not None:
    model._branch_var_ref = _resume_branch_var_ref.to(device)

# 6) Adaptive gradient clipping (AGC): clip iff ||g|| > c*||theta|| (ratio).
# EVA-блоки трансформероподобны -> c=0.1 (docstring: 0.01 — режим ResNet).
clipper = GradientClipper(c=0.1)
# τ-aware AGC: c_eff = c·(τ_ref/τ_l)^γ по τ-лестнице модели (per-layer map).
clipper.attach(model)
orig_seq_len = cfg.seq_len  # capture so recovery can undo OOM-driven seq_len shrink

# ─── P3: метакогнитивная инструментация (леджеры + KPI; обучение НЕ меняют) ───
# RegulatorLedger — парная identity-проба регуляторов forward-пути (только
# рекомендации, авто-отключение запрещено доктриной); BirthLedger — контрфакти-
# ческое изъятие новорождённых + MDL-цена; ParamVelocity — KPI эффективности
# параметров. Все три чекпойнтятся в envelope (state['ledger'/'birth_ledger'/
# 'pvel']) и восстанавливаются здесь. Ручки: probe-партии фиксируются на первом
# P3-eval (каждый 4-й канонический eval), pvel.sample — раз в log_interval.
from core.regulator_ledger import RegulatorLedger
from core.birth_ledger import BirthLedger
from core.param_velocity import ParamVelocity


def _p3_probe(x, y):
    """CE на фиксированной партии — нога identity/active леджера регуляторов."""
    with torch.no_grad():
        _p3st = int(globals().get('step', 0))
        h = model.embed_tokens(x)
        out, _, _, _ = model(h, None, global_state=None, adaptive=False,
                             tokens=x, step=_p3st)
        ce, _ = model.compute_losses(out, y, h_emb=h)
    return float(ce)


pvel = ParamVelocity(model)
reg_ledger = RegulatorLedger(model, cfg, probe=_p3_probe, per_round=3, dwell=3)
birth_ledger = BirthLedger()
model._birth_ledger = birth_ledger   # стэк-хуки (A)/(C) регистрируют рождения
_ck9 = globals().get('ckpt') or {}
if not starting_fresh and isinstance(_ck9, dict):
    reg_ledger.load_state_dict(_ck9.get('ledger'))
    birth_ledger.load_state_dict(_ck9.get('birth_ledger'))
    pvel.load_state_dict(_ck9.get('pvel'))
    print('  P3: instruments restored from checkpoint', flush=True)
print(f'  P3: instruments on — регуляторов в пробе: {len(reg_ledger.regs)}, '
      f'рождений в леджере: {len(birth_ledger.entries)}, '
      f'KPI-координат: {len(pvel.idx)}', flush=True)


In [ ]:
# @title 9. TRAINING LOOP (M28: ONLY core.adaptation intervenes — spectral-aligned aux, AGC incl. non-finite drop, plateau depth, mirror LR; no vetoes/guards in the loop)
import torch._dynamo
from torch.utils.checkpoint import CheckpointError as _CEr  # M22
def _d15(o):
    # B15 (F5-07): detach nested streaming state to CPU for checkpoint saving
    if torch.is_tensor(o):
        return o.detach().cpu()
    if isinstance(o, (list, tuple)):
        return [_d15(x) for x in o]
    return o
def _envelope(_step):
    # M42: ONE builder for the complete restart state — used by best.pt (the
    # ONLY checkpoint name, operator). Reads the CURRENT
    # loop variables at call time (cell-level globals).
    model.flush_control_pending()          # B16: deferred control writes -> durable
    _pn42 = {id(pp): n for n, pp in model.named_parameters()}
    import hashlib as _h42, json as _j42
    _tau_keys = ('tau_min','tau_max','T0','T_delay','delta_t','gate_tau_min',
                 'gate_tau_max','mem_tau_ref','llrd_gamma','seq_len','batch_size',
                 'n_layers','D','vocab')
    _cfg_fp42 = _h42.sha1(_j42.dumps({k: getattr(cfg, k, None) for k in _tau_keys},
                                      sort_keys=True, default=str).encode()).hexdigest()[:12]
    return {
        'step': int(_step), 'model': model.state_dict(), 'code_fp': codebook_fingerprint(model),
        'git_hash': GIT_HASH, 'cfg_fp': _cfg_fp42,
        'optimizer': optimizer.state_dict(),
        'param_names': [_pn42.get(id(p), 'external') for g in optimizer.param_groups for p in g['params']],
        'scheduler': scheduler.state_dict(),
        'best_val_loss': float(best_val_loss), 'cfg': cfg,
        'reasoning_enabled_step': reasoning_enabled_step,
        'active_depth': depth.active,
        'depth_state': depth.get_state(),
        'balancer': balancer.state_dict(),
        # P3: метакогнитивные инструменты (леджеры/KPI) — на обучение не влияют
        'ledger': reg_ledger.state_dict(),
        'birth_ledger': birth_ledger.state_dict(),
        'pvel': pvel.state_dict(),
        'branch_var_ref': (model._branch_var_ref.detach().cpu()
                           if getattr(model, '_branch_var_ref', None) is not None else None),
        'stream_idx': int(stream_idx), 'offset': int(offset),
        'rng': torch.get_rng_state(), 'data_rng': rng.get_state(),
        'stream_state': _d15(state), 'stream_gs': _d15(gs if gs is not None else None),
        'intent_stream': _d15(model._intent_stream)
                         if isinstance(getattr(model, '_intent_stream', None), list) else None,
        'cuda_rng': torch.cuda.get_rng_state() if device == 'cuda' else None,
    }

# T9.10: Drive-robust save. The Drive FUSE can silently swallow the 2.2 GB
# write (a 4840-step best downloaded as the 3960-step content). Write to the
# LOCAL runtime disk first (verified), then copy to the Drive via tmp+rename,
# verify (size + step + header/tail hash) with retries; the local copy stays
# as the session fallback (healed on the next start).
_LOCAL_CKPT_DIR = '/content/eva_ckpt_local'
try:
    import os as _os10
    from core.ckpt_io import heal_drive_from_local as _heal10
    if _heal10(_os10.path.join(SAVE_DIR, 'best.pt'),
               _os10.path.join(_LOCAL_CKPT_DIR, 'best.pt'),
               log=lambda m: print(m, flush=True)):
        print('[ckpt] T9.10: recovered a stale drive best.pt from the local copy',
              flush=True)
except Exception as _e10:
    print(f'[ckpt] T9.10 heal skipped: {_e10}', flush=True)

def _atomic_save(env, name):
    _p42 = os.path.join(SAVE_DIR, name)
    _l42 = os.path.join(_LOCAL_CKPT_DIR, name)
    try:
        from core.ckpt_io import save_best_robust as _sbr10
        if not _sbr10(env, _p42, _l42, step=int(env.get('step', -1)),
                      log=lambda m: print(m, flush=True)):
            print('[ckpt] WARNING: drive copy not verified - local fallback kept',
                  flush=True)
    except Exception as _e10b:
        print(f'[ckpt] T9.10 save error ({_e10b}); plain save fallback', flush=True)
        _t42 = _p42 + '.tmp'
        torch.save(env, _t42)
        import shutil as _sh42; _sh42.move(_t42, _p42)
    # T9.10b: tiny remote ground-truth file (small writes land reliably; the
    # 2.6 GB copy can be swallowed silently). The watchdog compares it with
    # the best.pt header from OUTSIDE the session.
    try:
        import json as _j10b, time as _t10b
        _st10b = int(env.get('step', -1))
        _meta10b = {'step': _st10b, 'val': float(env.get('best_val_loss', -1)),
                    'name': name, 'ts': _t10b.time()}
        _mp10b = os.path.join(SAVE_DIR, 'best_meta.json')
        with open(_mp10b + '.tmp', 'w') as _f10b:
            _j10b.dump(_meta10b, _f10b)
        os.replace(_mp10b + '.tmp', _mp10b)
        print(f'[ckpt] best_meta.json step={_st10b}', flush=True)
    except Exception as _e10c:
        print(f'[ckpt] best_meta write skipped: {_e10c}', flush=True)
    return _p42

torch._dynamo.config.suppress_errors = True
import time, os, math  # defensive: ensure available even if this cell runs standalone

# T9.12: remote-monitorable run — tee the console to the Drive via a
# background writer thread. The training NEVER blocks on the FUSE: the tee
# only enqueues; the writer appends+flushes and reopens on errors. The log
# (LOG_DIR/train_live.log) is what the operator reads remotely.
import sys as _sys12, threading as _th12, queue as _qu12
class _Tee12:
    def __init__(self, stream, q):
        self.stream, self.q = stream, q
    def write(self, s):
        try:
            self.stream.write(s)
        except Exception:
            pass
        try:
            self.q.put_nowait(s)
        except Exception:
            pass
        return len(s)
    def flush(self):
        try:
            self.stream.flush()
        except Exception:
            pass
def _log_writer12(path, q):
    # open/close PER WRITE: the Drive FUSE does not commit a file that stays
    # open (observed: train_live.log invisible to the Drive API for minutes);
    # close+fsync forces the commit. Writes are a few lines/minute — cheap.
    _d = os.path.dirname(path)
    while True:
        _s = q.get()
        try:
            os.makedirs(_d, exist_ok=True)
            with open(path, 'a', encoding='utf-8') as _f:
                _f.write(_s)
                _f.flush()
                try:
                    os.fsync(_f.fileno())
                except Exception:
                    pass
        except Exception:
            time.sleep(5)
_TRAIN_LOG_PATH = os.path.join(_LOCAL_CKPT_DIR, 'train_live.log')  # T9.12c: local disk —
# the Drive FUSE refused to commit appends to logs/train_live.log; the local
# file is copied to LOG_DIR at every eval (the val_history write path, proven).
_Q12 = _qu12.Queue(maxsize=100000)
_th12.Thread(target=_log_writer12, args=(_TRAIN_LOG_PATH, _Q12), daemon=True).start()
_sys12.stdout = _Tee12(_sys12.stdout, _Q12)
_sys12.stderr = _Tee12(_sys12.stderr, _Q12)
print(f'[log] tee -> {_TRAIN_LOG_PATH} (background writer)', flush=True)

stream_idx = resumed_stream_idx   # continue the data cursor (audit M12)
# B15-fix (T9-ревью R2): gs восстановлен из конверта в cell 8 — НЕ затирать
# безусловным None (иначе warm resume теряет cross-layer state).
if 'gs' not in globals():
    gs = None  # cross-layer stream state, threaded per step (audit M8)
offset = resumed_offset
if streams and stream_idx >= max(len(streams) - _hold_n, 1):
    # cursor saved under the pre-M13 1-file split could point INTO the new
    # 3-file hold-out — rotate to a fresh pool document instead
    stream_idx, offset = 0, 0
intent_state = None
tokens_seen = 0
t0 = time.time()
rng = torch.Generator(device='cpu').manual_seed(42)
if _resume_rng is not None:      # document-shuffle & dropout streams
    torch.set_rng_state(_resume_rng.cpu())   # continue where the ckpt left off (M12; .cpu() -- cell 8 loads with map_location=device, and RNG states MUST be CPU bytetensors)
if _resume_data_rng is not None:
    rng.set_state(_resume_data_rng.cpu())  # CPU-only contract (see above)
_oom_ckpt_step = None  # M35: when the ladder last forced recompute
_last_oom_step = -10**9  # M15: OOM-regrow cooldown (probe at most once /200 steps)

# gradalign: gradient-reactive governance loss for the MLP gate (0 = disabled).
cfg.gradalign_weight = 0.3

# Semantic Bridge is now IN-CORE (core/bridge.py): a per-layer SemanticBridge runs
# inside the model forward (train + inference). At every layer it emits a semantic
# vector, predicts the NEXT token's embedding (1 - cos aux loss, computed inside
# model.compute_losses -> aux_dict['bridge_conn']), and injects a persistent
# cross-layer stream back into the hidden state. Enabled via cfg.bridge_conn > 0
# (set in cell 4) at MODEL INIT time. No external head / extra param group => no
# StopIteration at checkpoint save (all bridge params live inside the model).
if getattr(cfg, 'bridge_conn', 0.0) > 0 and getattr(model, 'bridge', None) is not None:
    print(f'In-core SemanticBridge active (bridge_conn={cfg.bridge_conn}, '
          f'bridge_dim={model.bridge.bridge_dim}, params={sum(p.numel() for p in model.bridge.parameters()):,})')
elif getattr(cfg, 'bridge_conn', 0.0) > 0:
    print('WARNING: cfg.bridge_conn>0 but model.bridge is None -> set bridge_conn BEFORE model init.')
bridge_head = None


batch_size = getattr(cfg, 'batch_size', 1)

print(f'Git: running={GIT_HASH}')
try:
    _gh_now = subprocess.check_output(['git', '-C', DST, 'rev-parse', '--short', 'HEAD']).decode().strip()
    if _gh_now != GIT_HASH:
        print(f'  [WARN] stale clone — running {GIT_HASH} != HEAD {_gh_now}; force-pull cell 2 и перезапустите!')
except Exception:
    pass
print(f'Training: step {start_step} -> {cfg.max_steps}')
print(f'  ({cfg.max_steps - start_step} steps remaining)')
print(f'  batch={batch_size} seq={cfg.seq_len} -> tokens/step={batch_size * cfg.seq_len}')

try:
    for step in range(start_step, cfg.max_steps):
        model.train()

        if getattr(model, 'explicit_reasoning', False):
            model.reasoning_enabled_step = reasoning_enabled_step

        if streams:
            # Audit M8: rotate at DOCUMENT boundaries BEFORE the read. The old
            # `if offset == 0` never re-fired (get_batch rewound internally and
            # returned a non-zero offset), so one stream served the whole run
            # and banks/bus/reasoning never reset between documents. The LAST
            # stream is hold-out for eval → training pool is streams[:-1].
            _need = batch_size * cfg.seq_len + 1
            # M62: rotate the genre stream every `stream_chunk_steps` steps
            # (0 = legacy). The novelty machinery fires on DISTRIBUTION
            # SHIFTS: the exhaustion-only cadence gives none within budget
            # (streams are ~30-50M tokens -> a switch every ~1e5 steps), so
            # the lacuna/phantom/UCL chain had nothing to test on. The pick
            # excludes the current stream (a same-genre switch is a no-op).
            _chunk = int(getattr(cfg, 'stream_chunk_steps', 0) or 0)
            _rotate = (_chunk > 0 and step > 0 and step % _chunk == 0)
            if offset == 0 or offset + _need > streams[stream_idx].len or _rotate:
                _n_pick = max(len(streams) - _hold_n, 1)
                if _rotate and _n_pick > 1:
                    _p = int(torch.randint(0, _n_pick - 1, (1,), generator=rng).item())
                    stream_idx = _p if _p < stream_idx else _p + 1
                else:
                    stream_idx = torch.randint(0, _n_pick, (1,), generator=rng).item()
                offset = 0
                state = None
                intent_state = None
                gs = None
                # T7: единый холодный рестарт стримов (intent-поток, bus/salience,
                # bridge-stream) — раньше intent-поток молча переживал границу документа.
                model.reset_streams()
                if getattr(model, 'logit_cache', None) is not None:
                    model.logit_cache.cache.clear()  # new document ⇒ empty cache (decision #3)
                if getattr(model, 'memory_bank', None) is not None:
                    model.memory_bank.reset()
                if getattr(model, 'explicit_reasoning', False):
                    model.reset_reasoning()
            x, y, offset, _wrapped = streams[stream_idx].get_batch(cfg.seq_len, batch_size, offset)
        else:
            x = torch.randint(0, cfg.vocab, (batch_size, cfg.seq_len))
            y = torch.randint(0, cfg.vocab, (batch_size, cfg.seq_len))

        x, y = x.to(device), y.to(device)

        while True:
            try:
              with torch.autocast('cuda', dtype=_AMP_DTYPE, enabled=_USE_AMP):
                h = model.embed_tokens(x)
                out, state, gs, _ = model(h, state, global_state=gs, step=step, intent_state=intent_state, tokens=x)
                model.observe_output(model.lm_head(out))
                ce_loss, aux_dict = model.compute_losses(out, y, h_emb=h)
                # bridge_conn aux loss is produced in-core by model.compute_losses (SemanticBridge).

                # GRADALIGN now produced in-core: core.losses.compute_losses
                # builds aux_dict['gradalign'] from the block backward-hook target
                # (‖∂CE/∂mlp_out‖ per expert) with real cfg.gradalign_weight, and
                # LossBalancer.BYPASS_AUX backprops it directly (audit M5).
                break
            except torch.cuda.OutOfMemoryError:
                # Drop EVERY tensor bound by the failed attempt (and the
                # previous iteration): a retained forward graph (~4GB at
                # seq 256) makes even a seq=64 retry OOM (live incident
                # step 1052: rollback + retained graph = 22GB wall).
                h = out = ce_loss = aux_dict = None
                torch.cuda.empty_cache()
                state = None
                intent_state = None
                gs = None
                # M35: graded ladder — recompute first, then batch, then
                # window (batch was the dead end before: with B>1 the old
                # gate fell straight to the gc-dump raise).
                _laddered = True
                if not getattr(cfg, 'gradient_checkpointing', False):
                    cfg.gradient_checkpointing = True
                    _oom_ckpt_step = step          # M35: probe-back clock
                    print('  [OOM] gradient_checkpointing ON (recompute)', flush=True)
                elif batch_size > 1:
                    batch_size //= 2
                    print(f'  [OOM] recompute already on -> halving batch -> {batch_size}', flush=True)
                elif cfg.seq_len > 64:
                    cfg.seq_len //= 2
                    print(f'  [OOM] batch is 1 -> halving window -> {cfg.seq_len}', flush=True)
                else:
                    _laddered = False              # floor: fall through to the dump
                if _laddered:
                    x, y, offset, _w = streams[stream_idx].get_batch(cfg.seq_len, batch_size, offset)
                    # M35 FIX: the retry re-read the batch and FORGOT .to(device)
                    # — the first real ladder trip crashed with a device
                    # mismatch in the head's gather (live incident 2026-09-13).
                    x, y = x.to(device), y.to(device)
                    _last_oom_step = step
                    print(f'  [OOM] retry: seq_len={cfg.seq_len} batch={batch_size}')
                    continue
                import gc as _gc
                _gc.collect(); torch.cuda.empty_cache()
                _agg = {}
                for _o in _gc.get_objects():
                    try:
                        if torch.is_tensor(_o) and _o.is_cuda:
                            _k = (tuple(_o.shape), _o.dtype, 'grad' if _o.requires_grad else 'buf')
                            _agg[_k] = _agg.get(_k, 0) + _o.numel() * _o.element_size()
                    except Exception:
                        pass
                    if len(_agg) > 400:
                        break
                _top = sorted(_agg.items(), key=lambda kv: -kv[1])[:10]
                print('  [OOM] top CUDA tensor holders (shape, dtype, kind): bytes')
                for _k, _v in _top:
                    print(f'    {_k}  {_v/1e6:.0f}MB')
                raise

        tokens_seen += batch_size * cfg.seq_len
        intent_state = getattr(model, '_last_intent_state', None)

        depth.update(step)
        # Display loss (CE + raw aux sum); gradient uses the balancer.
        with torch.no_grad():
            disp_loss = ce_loss.item() + sum(float(v.item()) if isinstance(v, torch.Tensor) else float(v) for v in aux_dict.values()
                                            if isinstance(v, torch.Tensor))

        # Backward via principled balancer (spectral alignment) + AGC clip.
        if True:  # M28: backward unconditional; stabilization lives ONLY in core.adaptation
                        # B6 (GPT-#34): per-aux gradient geometry vs CE — who drives the model,
            # who fights it, who heats air (printed as name:ratio/cos where ratio
            # = ||g_aux||/||g_CE||). Costs a full backward pass per term, so it runs
            # only at log frequency and never touches .grad or the graph.
            # M48 (operator directive): ggeo REMOVED from the loop — it was
            # pure telemetry (never touched .grad), but its 8 retained
            # backward passes were the M47 incident's trigger and cost 9
            # extra forwards per log line. The capability stays where it
            # belongs: LossBalancer.grad_geometry in core.adaptation, for
            # offline use by the analyzer. The loop now contains exactly:
            # forward -> compute_losses -> balancer.backward -> tau-lr/AGC ->
            # step -> scheduler (plus memory/hardware handlers).
            # M22: B13b's backward freeze-wrap REMOVED — making recompute
            # path-dependent (freeze False in forward / True in backward
            # recompute) is precisely the 'different number of tensors
            # saved' class of CheckpointError. ggeo keeps its own wrap
            # (outside the checkpoint region). If a mismatch still occurs
            # at full size, self-heal: disable checkpointing, drop the
            # step, continue — never kill the run.
            # M64.12: the kill-switch measurement needs the LIVE graph -> before backward
            _ks = {}
            if balancer.kill is not None and step % max(cfg.log_interval, 1) == 0:
                _ks = balancer.measure_kill(ce_loss, aux_dict, model.parameters(), phase_model=model)
            try:
                balancer.backward(ce_loss, aux_dict, model.parameters(), phase_model=model,
                                 step=step)  # M64.4: align_every cadence
            except _CEr as _ce15:
                print('  [ckpt-fallback]', str(_ce15)[:120], '- checkpointing OFF for the run', flush=True)
                for _l in getattr(model, 'layers', []):   # M64.4: restore the hook
                    if hasattr(_l, '_ga_record'):
                        _l._ga_record = True
                cfg.gradient_checkpointing = False
                h = out = ce_loss = aux_dict = None
                optimizer.zero_grad(set_to_none=True)
                # M47 (live crash at 3190): the fallback MUST release the step
                # graph — the _cache_*/_pred_loss_term attrs still held LIVE
                # tensors of the failed step; the NEXT step's compute_losses
                # re-read _pred_loss_term and the backward died with
                # 'Trying to backward through the graph a second time'.
                model.release_step_graph()
                if device == 'cuda':
                    torch.cuda.empty_cache()
                continue
            # M64.8 (review fix): the liveness census must run AFTER the backward
            # and BEFORE zero_grad — the first landing placed it before the
            # backward (all 13 channels printed 0.0 on the real run).
            _gc = grad_census(model) if step % max(cfg.log_interval, 1) == 0 else None

            # U9 τ-aware AGC: effective clip ratio tightens as the τ-field matures
            # AGC с τ-aware порогом per-layer (c·(τ_ref/τ_l)^γ) — карта
            # построена clipper.attach(model) выше; τ-лестница обучается
            # медленно, перестроим карту раз в eval_interval шагов.
            if step % cfg.eval_interval == 0:
                clipper.attach(model)
            # B13 (audits 03 F3-03 + 04 F4-08, two agents independently):
            # grad-space LR multipliers MUST be applied BEFORE the AGC
            # clip — clipping first and scaling after voids the aux/CE
            # bound on the APPLIED update (measured 7.3-7.7x violations).
            # Per-layer learning rate distribution: scheduler ls_m (mirror
            # log-scale) × tau_config.lr_mult (τ-LLRD). Single source: apply_tau_lr.
            ls_mults = getattr(scheduler, '_ls_mult', None)
            apply_tau_lr(model, getattr(model, 'tau_config', None), ls_mults)
            clipper.clip(model.parameters())
            optimizer.step()
            optimizer.zero_grad(set_to_none=True)
            scheduler.step()
            # P3-4: KPI эффективности параметров (детерминированная подвыборка)
            if step % cfg.log_interval == 0:
                pvel.sample(model)
                if step % (4 * cfg.log_interval) == 0:
                    _p3r = pvel.report()
                    if _p3r:
                        print('  P3 pvel: ' + ' '.join(
                            f'{k}={v:.1e}' for k, v in sorted(_p3r.items())), flush=True)
            if getattr(model, 'explicit_reasoning', False):
                reasoning_enabled_step += 1
            # M15: probe regrow after an OOM-shrink, but at most once per 200
            # steps (a pure-OOM path must not run at 64 forever, nor oscillate)
            if cfg.seq_len < orig_seq_len and step - _last_oom_step > 200:
                _last_oom_step = step
                cfg.seq_len = min(cfg.seq_len * 2, orig_seq_len)
                print(f'  [OOM-recover] probe seq_len -> {cfg.seq_len}', flush=True)
            # M35: recompute is a ~30% tax; if the ladder enabled it and the
            # run then held stable for 500 steps, probe WITHOUT it. A fresh OOM
            # re-enables (clock resets) — the ladder is a ratchet with memory.
            if (getattr(cfg, 'gradient_checkpointing', False) and _oom_ckpt_step is not None
                    and step - _oom_ckpt_step > 500):
                cfg.gradient_checkpointing = False
                _oom_ckpt_step = None
                print('  [ckpt-probe] 500 stable steps -> continuing WITHOUT recompute', flush=True)

        model.release_step_graph()   # M21: the _cache_*/_cached_* attrs pin
        # the whole step graph between steps (gc-proven 27GB leak).
        if state is not None:
            state = tuple(
                tuple(t.detach() if isinstance(t, torch.Tensor) else t for t in s)
                if s else None for s in state
            )
        if intent_state is not None:
            intent_state = intent_state.detach()
        gs = gs.detach() if gs is not None else None

        if step % cfg.log_interval == 0:
            dt = time.time() - t0
            tok_s = tokens_seen / max(dt, 1e-8)
            lr = scheduler.get_last_lr()[0]
            mem_gb = torch.cuda.max_memory_allocated() / (1024 ** 3) if device == 'cuda' else 0
            live_gb = torch.cuda.memory_allocated() / (1024 ** 3) if device == 'cuda' else 0  # D6 telemetry: live vs peak
            # Effective gate amplitude (normalized in-core by running RMS EMA,
            # blind to raw ‖w_intent‖ growth): ~1.0 healthy, >2 sustained = danger.
            intent_eff = float('nan')
            try:
                _i = [getattr(_l.mirror, '_cached_ig_eff', None) for _l in model.layers]
                if _i and all(x is not None for x in _i):
                    intent_eff = sum(_i) / len(_i)
            except Exception:
                pass
            mod_scl = 0.0
            mod_scl_std = 0.0
            try:
                if any(l.mirror.bridge_glu_net is not None for l in model.layers):
                    _bg = torch.cat([l.mirror._last_mlp_mod.flatten() for l in model.layers])
                    mod_scl, mod_scl_std = _bg.mean().item(), _bg.std().item()
                else:
                    mod_scl = torch.stack([torch.sigmoid(l.mirror.mod_scale_mlp).mean()
                                       for l in model.layers]).mean().item()
            except Exception:
                pass
            # MLP-collapse diagnostics: raw MLP output norm, usefulness gate, maturation gate.
            mlp_out_n = usef_m = usef_s = mat_g = mat_gmin = mat_gmax = float('nan')
            try:
                with torch.no_grad():
                    if all(getattr(l, '_cache_mlp_out', None) is not None for l in model.layers):
                        mlp_out_n = torch.stack([l._cache_mlp_out.detach().norm() for l in model.layers]).mean().item()
                    # M64.6 (M63-E): the MEAN is structurally ~0.5 (the usefulness
                    # sigmoid is median-centered) — log the STD, which carries the signal.
                    _us = torch.stack([l.mirror._cached_usefulness.detach().flatten() for l in model.layers])
                    usef_m = _us.mean().item()
                    usef_s = _us.std().item()
                    if getattr(model, 'maturation', None) is not None:
                        _g = model.maturation.gate.detach()
                        mat_g, mat_gmin, mat_gmax = _g.mean().item(), _g.min().item(), _g.max().item()
            except Exception:
                pass
            # Memory Bank diagnostics
            mb_diag = ''
            if getattr(model, 'memory_bank', None) is not None:
                try:
                    _mb = model.memory_bank.get_diagnostics()
                    mb_diag = f" L1={_mb['l1_write_idx']} L2={_mb['l2_write_idx']}({_mb['l2_consumed']}c) L3={_mb['l3_n_concepts']}({_mb['l3_n_births']}b) scale={_mb['mem_scale']:.3f}"
                except Exception:
                    pass
            # Merge aux_dict + _cached_losses (τ-gating diagnostics live there)
            _lc = getattr(model, '_cached_losses', {})
            _merged = dict(aux_dict)
            for _k, _v in _lc.items():
                if _k not in _merged:
                    _merged[_k] = _v
            aux_str = ' '.join(f'{k}={v:.4f}' for k, v in sorted(_merged.items()) if abs(v.item() if isinstance(v, torch.Tensor) else float(v)) > 1e-6)
            # M64.4 (R3): the balancer cadence telemetry — without it the
            # align_every A/B is indistinguishable in the log.
            _bal_s = getattr(balancer, 'scale_ema', None)
            _bal_sc = getattr(balancer, 'last_scale', None)
            _bal_c = getattr(balancer, 'last_align_cos', None)  # M64.4r3: survives cheap steps
            _bal = (f'bal_a={getattr(balancer, "n_align", 0)} '
                    f'bal_b={getattr(balancer, "n_balance", 0)} '
                    f'bal_s={"None" if _bal_s is None else round(float(_bal_s), 5)} '
                    f'bal_sc={"None" if _bal_sc is None else round(float(_bal_sc), 5)} '
                    f'bal_cos={"None" if _bal_c is None else round(float(_bal_c), 4)}  ')
            print(f'step={step:>6}  loss={disp_loss:.4f}  ce={ce_loss.item():.4f}  '
                  f'mod_mlp={mod_scl:.3f} mod_std={mod_scl_std:.3f} lr={lr:.2e}  tok/s={tok_s:.0f}  {_bal}'
                  f'mem={mem_gb:.1f}GB live={live_gb:.1f} d={depth.active}  intent_eff={intent_eff:.4f}  mlp_out={mlp_out_n:.1f} usef={usef_m:.3f}/{usef_s:.3f}  '
                  f'mat={mat_g:.3f}[{mat_gmin:.3f},{mat_gmax:.3f}]{mb_diag}')
            if aux_str:
                print(f'  aux: {aux_str}')
            # M64.2r2: telemetry must NEVER kill the run — the quantile
            # device crash at step 7425 lived exactly here (CPU tests can't
            # see it). Every telemetry call is crash-proof.
            try:
                _tel = model.head_telemetry() if hasattr(model, 'head_telemetry') else {}
                if _tel:
                    print('  head: ' + ' '.join(
                        (f'{k}={v:.4f}' if isinstance(v, float) else f'{k}={v}') for k, v in _tel.items()))
            except Exception as _te2:
                print(f'  head: skipped ({str(_te2)[:60]})')
            # M64.8: the telemetry batch + the grad census
            try:
                _tt = training_telemetry(model)
                _tt.update(_gc or {})
                if _tt:
                    print('  tele: ' + ' '.join(
                        f'{k}={v:.4g}' if isinstance(v, float) else f'{k}={v}'
                        for k, v in _tt.items()))
            except Exception as _te:
                print(f'  tele: skipped ({str(_te)[:60]})')
            if _ks:
                print('  ks: ' + ' '.join(f'{k}={v:.4g}' for k, v in _ks.items())
                      + ' | ' + ' '.join(f'{k}={v}' for k, v in balancer.kill.stats().items()))
            if device == 'cuda':
                torch.cuda.reset_peak_memory_stats()

        # M42 flush (the rolling M42 file is gone — operator: the checkpoint
        # name is best, as before). Every 495 steps flush everything that is
        # NOT continuity-critical for training: logit cache contents, mirror
        # stream caches + diag buffers (M41 shrink), observers re-warm; the
        # allocator cache is returned to the driver. The A100 cycle showed
        # reserved creeping +1.7GB per ggeo window while live stayed flat —
        # fragmentation from the periodic spikes; flushing bounds it.
        if step > 0 and step % 495 == 0:
            model.reset_cache()
            if device == 'cuda':
                torch.cuda.empty_cache()
            gc.collect()
            print(f'  [flush] step={step}: caches flushed '
                  f'(alloc {torch.cuda.memory_allocated()/1e9:.1f} / '
                  f'reserved {torch.cuda.memory_reserved()/1e9:.1f} GB)', flush=True)
        # M49 (operator): EARLY MEASUREMENT EVALS. Before 1045 there was no
        # val signal and no val-clean checkpoint at all. Now: additional
        # evals every `eval_early_every` steps until `eval_early_until`
        # (defaults 250/3000), which write best.pt/val_history but
        # DO NOT touch the controllers — depth-plateau and LR damping stay on
        # the canonical cadence (dynamics identical, only observability
        # grows). Cache hygiene after eval is unchanged and already correct:
        # snapshot/restore brings the training document's streaming state
        # back (M33 doctrine — the bank/UCL/mirror chain must NOT be reset),
        # the logit cache is cleared, allocator cache returned.
        _canonical_eval = (step > 0 and step % cfg.eval_interval == 0)
        _early_eval = (step > 0 and not _canonical_eval
                       and step < getattr(cfg, 'eval_early_until', 3000)
                       and step % getattr(cfg, 'eval_early_every', 250) == 0)
        if _canonical_eval or _early_eval:
            model.eval()
            _lcache = getattr(model, 'logit_cache', None)
            if _lcache is not None:
                _lcache.cache.clear()  # eval isolation for the cache list (decision #3)
            _rt_snap = model.snapshot_runtime_buffers()  # val must not touch train working memory (M8)
            # M32: chain reset moved PER hold-out document (see loop below).
            # Hold-out eval (audit M13): average over the last _hold_n FILES,
            # each read from its 3/4-region (no wrapped re-read, no train
            # overlap). Per-file batch budget 100//_hold_n keeps total eval
            # cost and the val scale history comparable to the M8 era.
            # M32: windows CARRY the document's VSA state and deliberation
            # chain (train-identical semantics); the boundary is the FILE.
            eval_pool = []
            if streams:
                eval_pool = streams[-_hold_n:] if _hold_n else [streams[-1]]
                if not any(getattr(s, 'len', 0) >= batch_size * cfg.seq_len + 1
                           for s in eval_pool):
                    eval_pool = [streams[0]]  # tiny corpus: M8 fallback
            val_loss = 0.0
            n_val = 0
            _val_ok = False
            for eval_stream in eval_pool:
                if getattr(eval_stream, 'len', 0) < batch_size * cfg.seq_len + 1:
                    continue  # file too short for even one batch
                # M32: fresh chain + fresh stream state at the hold-out FILE
                # boundary, then carry across the file's windows — exactly the
                # training sampler's document discipline.
                if getattr(model, 'explicit_reasoning', False):
                    model.reset_reasoning()
                if getattr(model, 'memory_bank', None) is not None:   # M33: fresh bank per doc
                    model.memory_bank.reset()
                model.reset_streams()   # T7: intent/bus/salience/bridge — холодный старт файла
                vst = vgs = None
                _vw_rec = []
                with torch.no_grad():
                    voff = max(eval_stream.len // 4, batch_size * cfg.seq_len + 1)
                    _vmax = max(min(100 // max(_hold_n, 1),
                                    eval_stream.len // (batch_size * cfg.seq_len)), 1)
                    for _ in range(_vmax):
                        vx, vy, voff, _vw = eval_stream.get_batch(cfg.seq_len, batch_size, voff)
                        if _vw:
                            break  # end of hold-out region; no wrapped re-read (audit M8)
                        vx, vy = vx.to(device), vy.to(device)
                        h = model.embed_tokens(vx)
                        # T4 parity-eval: step=step — temper/lacuna-каналы как на train
                        out, vst, vgs, _ = model(h, vst, global_state=vgs, adaptive=False, tokens=vx, step=step)
                        ce, _ = model.compute_losses(out, vy, h_emb=h)
                        val_loss += ce.item()
                        n_val += 1
                        _tel_w = model.head_telemetry() if hasattr(model, 'head_telemetry') else {}
                        _vw_rec.append({'file': getattr(eval_stream, 'path', '?'),
                                        'offset': int(voff - cfg.seq_len), 'ce': ce.item(),
                                        'sat': float(_tel_w.get('sat', float('nan'))),
                                        'conflict': float(_tel_w.get('conflict', float('nan')))})  
                        _val_ok = True
            if _vw_rec:
                _wce = sorted(_vw_rec, key=lambda r: r['ce'], reverse=True)
                _wmean = sum(r['ce'] for r in _vw_rec) / len(_vw_rec)
                print(f'  EVAL windows: n={len(_vw_rec)} ce_mean={_wmean:.4f} ce_max={_wce[0]["ce"]:.4f} '
                      f'sat_max={max(r["sat"] for r in _vw_rec):.3f} conflict_max={max(r["conflict"] for r in _vw_rec):.3f}')
                print('  EVAL worst 3:', '; '.join(
                    f'{r["file"]}@{r["offset"]} ce={r["ce"]:.2f}' for r in _wce[:3]))
            if _val_ok:
                val_loss /= n_val
                val_ppl = math.exp(val_loss) if val_loss < 20 else float('inf')
                print(f'  EVAL step={step}: val_loss={val_loss:.4f} val_ppl={val_ppl:.2e} (n={n_val})', flush=True)
                # ─── P3: метакогнитивные измерения (каждый 4-й канонический eval) ───
                if step % (4 * cfg.eval_interval) == 0:
                    try:
                        if not reg_ledger.batches:
                            _pb1 = max(eval_stream.len // 4,
                                       batch_size * cfg.seq_len + 1)
                            _bx1, _by1, _, _vw1 = eval_stream.get_batch(
                                cfg.seq_len, batch_size, _pb1)
                            _bx2, _by2, _, _vw2 = eval_stream.get_batch(
                                cfg.seq_len, batch_size, _pb1 + batch_size * cfg.seq_len)
                            if not (_vw1 or _vw2):
                                reg_ledger.batches = [
                                    (_bx1.to(device), _by1.to(device)),
                                    (_bx2.to(device), _by2.to(device))]
                                print(f'  P3: probe-партии зафиксированы '
                                      f'({batch_size}x{cfg.seq_len})', flush=True)
                        if reg_ledger.batches:
                            _p3d = reg_ledger.measure_round(model)
                            _p3sug = reg_ledger.suggestions()
                            print('  P3 ledger: ' + ' '.join(
                                f'{k}={v:+.4f}' for k, v in sorted(_p3d.items())), flush=True)
                            _sr = (reg_ledger.sigma_re
                                   if reg_ledger.sigma_re is not None else float('nan'))
                            _sb = (reg_ledger.sigma_b
                                   if reg_ledger.sigma_b is not None else float('nan'))
                            print(f'  P3 noise: sigma_re={_sr:.5f} '
                                  f'sigma_b={_sb:.4f}'
                                  + (f'  DORMANT-кандидаты: {", ".join(_p3sug)}'
                                     if _p3sug else ''), flush=True)
                            _p3b = birth_ledger.measure(
                                model, reg_ledger.probe, reg_ledger.batches[0],
                                step, batch_size * cfg.seq_len)
                            if _p3b:
                                print(f'  P3 births: {_p3b} '
                                      f'stats={birth_ledger.stats()}', flush=True)
                    except Exception as _p3e:
                        print(f'  P3: измерение не удалось '
                              f'({type(_p3e).__name__}: {_p3e})', flush=True)
                if _canonical_eval:      # M49: controls stay on the canonical cadence
                    depth.update(step, val_loss)
                    scheduler.report_val_loss(val_loss)
                # B15 (audit 05 F5-06): restore the PRE-EVAL runtime state BEFORE
                # the save — model.state_dict() used to capture 22 eval-mutated
                # persistent keys (UCL concepts, bridge stream) into the CLEAN artifact.
                model.restore_runtime_buffers(_rt_snap)
                _lc15 = getattr(model, 'logit_cache', None)
                if _lc15 is not None:
                    _lc15.cache.clear()
                if val_loss < best_val_loss:
                    best_val_loss = val_loss
                    _atomic_save(_envelope(step), 'best.pt')
                    print(f'  EVAL saved best.pt (val_loss={val_loss:.4f}) step={step}', flush=True)
                else:
                    print(f'  EVAL no-improve (best={best_val_loss:.4f})', flush=True)
                # Оператор: на eval сохраняется ТОЛЬКО best.pt (один файл);
                # rolling eval_last.pt удалён.
                # val log only — best.pt is written above on improvement
                # (operator: the checkpoint name is best).
                try:
                    with open(os.path.join(SAVE_DIR, 'val_history.jsonl'), 'a') as _vh42:
                        _vh42.write(json.dumps({'step': int(step), 'val_loss': float(val_loss),
                                                 'best_val_loss': float(best_val_loss),
                                                 'ce_max': _wce[0]['ce'],
                                                 'sat_max': max(r['sat'] for r in _vw_rec),
                                                 'conflict_max': max(r['conflict'] for r in _vw_rec)}) + '\n')
                except Exception:
                    pass
                # T9.10b: cheap header check at every eval - if the drive best.pt
                # is behind the local fallback (a swallowed save), re-copy it.
                try:
                    from core.ckpt_io import heal_drive_from_local as _heal10b
                    _heal10b(os.path.join(SAVE_DIR, 'best.pt'),
                             os.path.join(_LOCAL_CKPT_DIR, 'best.pt'),
                             log=lambda m: print(m, flush=True))
                except Exception as _e10d:
                    print(f'[ckpt] heal check skipped: {_e10d}', flush=True)
                # T9.12c: publish the local console log to the Drive at every eval
                try:
                    import shutil as _sh12c
                    _sh12c.copyfile(os.path.join(_LOCAL_CKPT_DIR, 'train_live.log'),
                                    os.path.join(LOG_DIR, 'train_live.log'))
                except Exception:
                    pass
            else:
                print(f'  EVAL step={step}: NO HOLD-OUT DATA (streams empty) - skipping val_loss', flush=True)
                print(f'  NOTE: training appears to be on RANDOM data (no token_stream_*.bin found)', flush=True)
                depth.update(step, None)
            _lcache = getattr(model, 'logit_cache', None)
            if _lcache is not None:
                _lcache.cache.clear()
            model.restore_runtime_buffers(_rt_snap)  # (audit M8 eval isolation)
            model.train()
            gc.collect()
            if device == 'cuda':
                torch.cuda.empty_cache()

        # Checkpoint policy (audit M12, single file): best.pt is the ONLY
        # checkpoint. Val-improving EVAL writes carry the complete restart
        # state (weights, optimizer+param_names, scheduler, depth &
        # balancer statistics, data cursor, RNG streams); a step-0 seed
        # written in setup backs the first val eval.

except KeyboardInterrupt:
    print('Interrupted - saving best.pt...')
    # Operator: the checkpoint name is best, as before M42 (no rolling file).
    # The interrupt state overwrites best.pt — the historical behavior —
    # so FORCE_FRESH=False always finds a complete state.
    _env42i = _envelope(step)
    _atomic_save(_env42i, 'best.pt')
    print(f'Saved interrupt state (step {step}) -> best.pt')

print('Training complete!')
